# Feature Engineering

In this notebook, we process the cleaned datasets to obtain the properly encoded final features.

In [62]:
import pandas as pd
from langchain_dartmouth.llms import ChatDartmouthCloud

admissions = pd.read_csv("../data/derived/admissions.csv")
interviews = pd.read_csv("../data/derived/interviews.csv")
outcomes = pd.read_csv("../data/derived/outcomes.csv")

In [63]:
admissions

,Student ID,Waitlisted,Birthdate,Marital or Partnership Status,Do you have children?,Children Information,Citizenship Status,Race,Ipeds_Race,Sex,Are you being sponsored,Job 1 Organization,Job 1 Compensation,Job 1 Bonus,Job #1 Industry Code,Job 1 Title,Round,Grad Year
0,240044010110126,No,1990-05-30,Will marry before matriculation,0.0,NaN,US Citizen,White,White,F,0.0,Cambridge Associates,60000.0,7000.0,Financial Services - Investment Management/Res...,Senior Client Operations Associate,"Early Action (October 5, 2016)",2019
1,220044010110050,No,1988-12-16,Married Partnership,1.0,Luis Enrique Acosta Elias 1 year,Foreign National,NaN,Not US Citizen or PR,M,0.0,Emerson Climate Technologies,25800.0,2838.0,Manufacturing,Project Leader,Early Action (Oct.-07-2015),2018
2,220044010110038,No,1990-08-28,Nonmarried Partnership,0.0,NaN,US Citizen,White,White,M,0.0,"Deloitte Consulting, LLP",83000.0,7500.0,Consulting - Strategy/Management,Federal Strategy and Operations Consultant,Early Action (Oct.-07-2015),2018
3,540016052438215,No,1992-01-07,Married Partnership,0.0,NaN,US Citizen,White,White,F,0.0,Riley Home,90000.0,0.0,Retail,Merchandise Planner,"Round 2 (January 3, 2022)",2024
4,540016052438313,No,1994-10-11,Single,0.0,NaN,US Citizen,White,White,F,0.0,Smart Air Mongolia LLC,0.0,0.0,Consumer Goods - Electronics,Executive Director,"Round 1 (September 27, 2021)",2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1424,540016052438175,Yes,1993-05-24,NaN,0.0,NaN,US Citizen,White,White,F,0.0,Darktrace,100000.0,50000.0,Technology - Software,Director of Sales - US West,"Round 1 (September 27, 2021)",2024
1425,540016052438112,No,1992-12-15,Nonmarried Partnership,0.0,NaN,US Citizen,White,White,F,0.0,NewYork-Presbyterian,111300.0,5500.0,Healthcare - Providers/Health Services,"Head of Digital, Division of Community & Popul...","Round 2 (January 3, 2022)",2024
1426,330038051130754,Yes,1990-04-21,Nonmarried Partnership,0.0,NaN,US Citizen,White,White,F,0.0,NBCUniversal,71750.0,NaN,Advertising,Senior Analyst,Early Action (Oct.-07-2015),2018
1427,540016051186067,No,1993-08-11,Single,0.0,NaN,Foreign National,NaN,Not US Citizen or PR,F,0.0,State Farm Canada - Desjardins Insurance,46000.0,4000.0,Financial Services - Other,Advisory Analyst,"Round 2 (January 7, 2019)",2021


In [64]:
admissions_features = pd.DataFrame()
admissions_features["Student ID"] = admissions["Student ID"]
admissions_features["Age at Graduation"] = (
    pd.to_datetime(admissions["Birthdate"]).dt.year
    - pd.to_datetime(admissions["Grad Year"]).dt.year
)

In [65]:
admissions_features = pd.concat(
    [admissions_features, pd.get_dummies(admissions["Marital or Partnership Status"])],
    axis="columns",
)

In [66]:
admissions_features["Has Children"] = admissions["Do you have children?"] > 0

In [67]:
# TODO: How to code the round?
admissions["Round"].value_counts()

Round
Early Action (October 4, 2017)       119
Early Action (October 5, 2016)       111
Early Action (Oct.-07-2015)          111
Round 2 (January 7, 2019)             98
Round 2 (January 6, 2020)             98
Round 1 (September 24, 2018)          92
Round 1 (October 7, 2019)             84
Round 2 (January 4, 2021)             80
Round 1 (September 28, 2020)          77
Round 1 (September 27, 2021)          77
January Round (Jan.-06-2016)          63
Round 2 (January 3, 2022)             62
January (January 7, 2018)             59
January Round (January 4, 2017)       56
November (November 1, 2017)           36
November Round (Nov.-04-2015)         32
November Round (November 2, 2016)     26
Round 4 (June 1, 2020)                14
Round 2 CGSM (January 5, 2020)        12
CGSM October (October 15, 2016)       12
CGSM January (January 5, 2017)        10
CGSM January (January 5, 2016)        10
Round 1 CGSM (October 15, 2018)        8
Round 2 CGSM (January 5, 2019)         8
CGSM Octob

In [68]:
admissions_features["Is Citizen"] = admissions["Citizenship Status"] == "US Citizen"

In [69]:
def dummy_encode_race(race_col) -> pd.DataFrame:
    dummies = pd.DataFrame()
    # Define all possible race categories
    race_categories = [
        "White",
        "Asian",
        "Black or African American",
        "Hispanic, Latino, or Spanish Origin",
        "Middle Eastern or North African",
        "Native Hawaiian or Other Pacific Islander",
        "American Indian or Alaska Native",
    ]

    # Create a new column for each race category
    for race in race_categories:
        dummies[f"Race_{race}"] = race_col.str.contains(race, regex=False)

    dummies = dummies.fillna(False)
    return dummies


admissions_features = pd.concat(
    [admissions_features, dummy_encode_race(admissions["Race"])], axis="columns"
)

/var/folders/hh/0nzphmq50n527kqkm_58h6700000gp/T/ipykernel_47432/746830609.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dummies = dummies.fillna(False)


In [70]:
admissions_features = pd.concat(
    [
        admissions_features,
        pd.get_dummies(admissions["Ipeds_Race"], prefix="IPEDS_Race_"),
    ],
    axis="columns",
)

In [71]:
admissions_features = pd.concat(
    [
        admissions_features,
        pd.get_dummies(admissions["Sex"]),
    ],
    axis="columns",
)

In [75]:
admissions_features["Is Sponsored"] = admissions["Are you being sponsored"] == 1.0

In [76]:
admissions_features["Employer"] = admissions["Job 1 Organization"]
admissions_features["Job Title"] = admissions["Job 1 Title"]
admissions_features["Total Compensation"] = admissions["Job 1 Compensation"].fillna(
    0
) + admissions["Job 1 Bonus"].fillna(0)

In [77]:
admissions_features

,Student ID,Age at Graduation,Married Partnership,Nonmarried Partnership,Single,Will marry before matriculation,Has Children,Is Citizen,Race_White,Race_Asian,...,IPEDS_Race__Multi-Race,IPEDS_Race__Multi-race,IPEDS_Race__Not US Citizen or PR,IPEDS_Race__White,F,M,Is Sponsored,Employer,Job Title,Total Compensation
0,240044010110126,20,False,False,False,True,False,True,True,False,...,False,False,False,True,True,False,False,Cambridge Associates,Senior Client Operations Associate,67000.0
1,220044010110050,18,True,False,False,False,True,False,False,False,...,False,False,True,False,False,True,False,Emerson Climate Technologies,Project Leader,28638.0
2,220044010110038,20,False,True,False,False,False,True,True,False,...,False,False,False,True,False,True,False,"Deloitte Consulting, LLP",Federal Strategy and Operations Consultant,90500.0
3,540016052438215,22,True,False,False,False,False,True,True,False,...,False,False,False,True,True,False,False,Riley Home,Merchandise Planner,90000.0
4,540016052438313,24,False,False,True,False,False,True,True,False,...,False,False,False,True,True,False,False,Smart Air Mongolia LLC,Executive Director,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1424,540016052438175,23,False,False,False,False,False,True,True,False,...,False,False,False,True,True,False,False,Darktrace,Director of Sales - US West,150000.0
1425,540016052438112,22,False,True,False,False,False,True,True,False,...,False,False,False,True,True,False,False,NewYork-Presbyterian,"Head of Digital, Division of Community & Popul...",116800.0
1426,330038051130754,20,False,True,False,False,False,True,True,False,...,False,False,False,True,True,False,False,NBCUniversal,Senior Analyst,71750.0
1427,540016051186067,23,False,False,True,False,False,False,False,False,...,False,False,True,False,True,False,False,State Farm Canada - Desjardins Insurance,Advisory Analyst,50000.0
